In [1]:
import polars as pl
import os

# CSV 파일의 절대경로
# Notebook이 analysis/ 폴더에 있으므로 부모 디렉토리의 output/ 찾기
csv_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'output', 'all_trials_timeseries.csv')

# 혹은 현재 파일 위치에서 상대경로로 찾기 (더 안정적)
# analysis.ipynb는 analysis 폴더에 있으므로
import sys
from pathlib import Path
notebook_dir = Path('.').resolve()
csv_path = notebook_dir.parent / 'output' / 'all_trials_timeseries.csv'

print(f"Current dir: {notebook_dir}")
print(f"CSV Path: {csv_path}")
print(f"Exists: {csv_path.exists()}")

# CSV 읽기
df = pl.read_csv(csv_path)

print("\n" + "="*80)
print("STEP vs NONSTEP 분석 (Subject-Velocity-Trial 단위)")
print("="*80)

# Trial별 step_TF 결정 (각 trial은 하나의 step_TF 값을 가짐)
trial_level = df.select(['subject', 'velocity', 'trial', 'step_TF']).unique().sort(
    ['subject', 'velocity', 'trial']
)

print("\n[1] Trial별 Step/Nonstep 분포")
print("-" * 50)
trial_step_dist = trial_level.group_by('step_TF').agg(
    pl.len().alias('trial_count')
).with_columns(
    ratio_percent = (pl.col('trial_count') / pl.col('trial_count').sum() * 100).round(2)
).sort('step_TF', descending=True)
print(trial_step_dist)

print("\n[2] 피험자별 Step/Nonstep Trial 개수")
print("-" * 50)
subject_step_count = trial_level.group_by(['subject', 'step_TF']).agg(
    pl.len().alias('trial_count'),
    pl.col('velocity').n_unique().alias('velocity_type_count')
).sort(['subject', 'step_TF'])
print(subject_step_count)

print("\n[3] 피험자별 총 Trial 개수")
print("-" * 50)
subject_total = trial_level.group_by('subject').agg(
    pl.len().alias('total_trial_count'),
    pl.col('velocity').n_unique().alias('velocity_count')
).sort('subject')
print(subject_total)

print("\n[4] 상세 Trial 목록 (Subject-Velocity-Trial-Step)")
print("-" * 50)
detailed = trial_level.sort(['subject', 'velocity', 'trial'])
print(detailed)

print("\n[5] 피험자별 Step/Nonstep Trial 개수 Summary")
print("-" * 50)
subject_summary = trial_level.group_by(['subject', 'step_TF']).agg(
    pl.len().alias('trial_count')
).sort(['subject', 'step_TF'])
print(subject_summary)

Current dir: C:\Users\Alice\OneDrive - 청주대학교\근전도 분석 코드\replace_V3D\analysis
CSV Path: C:\Users\Alice\OneDrive - 청주대학교\근전도 분석 코드\replace_V3D\output\all_trials_timeseries.csv
Exists: True

STEP vs NONSTEP 분석 (Subject-Velocity-Trial 단위)

[1] Trial별 Step/Nonstep 분포
--------------------------------------------------
shape: (2, 3)
┌─────────┬─────────────┬───────────────┐
│ step_TF ┆ trial_count ┆ ratio_percent │
│ ---     ┆ ---         ┆ ---           │
│ str     ┆ u32         ┆ f64           │
╞═════════╪═════════════╪═══════════════╡
│ step    ┆ 1           ┆ 33.33         │
│ nonstep ┆ 2           ┆ 66.67         │
└─────────┴─────────────┴───────────────┘

[2] 피험자별 Step/Nonstep Trial 개수
--------------------------------------------------
shape: (2, 4)
┌─────────┬─────────┬─────────────┬─────────────────────┐
│ subject ┆ step_TF ┆ trial_count ┆ velocity_type_count │
│ ---     ┆ ---     ┆ ---         ┆ ---                 │
│ str     ┆ str     ┆ u32         ┆ u32                 │
╞═══════